# Trabajo Final Integrador — Detección temprana de riesgo académico
**Doctorado en Informática – UTN FRRe/UNNE/UNAM**  
Curso: Introducción al Aprendizaje de Máquina  
Integrantes: Stella Gerzel · Juan Manuel Goya · Gustavo Ebel

---
Este notebook integra el trabajo completo. Está organizado en las siguientes secciones:

1. Contexto y problemática
2. Descripción y análisis exploratorio de datos
3. Preparación del dataset para modelado
4. Entrenamiento y evaluación de modelos
5. Importancia de variables
6. Ajuste del umbral de decisión
7. Conclusiones


## Librerías y carga del dataset

In [ ]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
                             f1_score, classification_report)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
print('Librerías OK')

In [ ]:
# Carga del dataset
from google.colab import files
uploaded = files.upload()

ruta = next(iter(uploaded))
df = pd.read_excel(ruta, sheet_name='Datos_modelado_base')
df_w = df.copy()

# Separar predictoras del target
X_base = df_w.drop(columns=['nota']).copy()

# 3 targets binarios
y1 = df_w['nota'].map({'A':0,'R':1,'U':1})
mask_p2 = df_w['nota'].isin(['A','R'])
X_p2 = X_base[mask_p2].reset_index(drop=True)
y2   = df_w.loc[mask_p2,'nota'].map({'A':0,'R':1}).reset_index(drop=True)
mask_p3 = df_w['nota'].isin(['A','U'])
X_p3 = X_base[mask_p3].reset_index(drop=True)
y3   = df_w.loc[mask_p3,'nota'].map({'A':0,'U':1}).reset_index(drop=True)

# Splits estratificados 75/25
X1_tr,X1_te,y1_tr,y1_te = train_test_split(X_base,y1,test_size=0.25,stratify=y1,random_state=42)
X2_tr,X2_te,y2_tr,y2_te = train_test_split(X_p2,  y2,test_size=0.25,stratify=y2,random_state=42)
X3_tr,X3_te,y3_tr,y3_te = train_test_split(X_p3,  y3,test_size=0.25,stratify=y3,random_state=42)

# Columnas por tipo
cols_cat = X_base.select_dtypes('object').columns.tolist()
cols_num = X_base.select_dtypes('number').columns.tolist()

print(f'Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Predictoras: {X_base.shape[1]} ({len(cols_cat)} categóricas, {len(cols_num)} numéricas)')
print(f'Train P1:{len(X1_tr)} | P2:{len(X2_tr)} | P3:{len(X3_tr)}')

---
## 1. Contexto y problemática

> ⚠️ **PENDIENTE — Stella completa esta sección con el texto de contexto institucional del notebook 01**

---
## 2. Descripción y análisis exploratorio de datos

> ⚠️ **PENDIENTE — Stella copia aquí las celdas principales del EDA: distribución del target, análisis de faltantes, ranking exploratorio de variables y gráficos del notebook 02**

---
## 3. Preparación del dataset para modelado

> ⚠️ **PENDIENTE — Stella copia aquí el resumen de decisiones de descarte del notebook 03: tabla de columnas eliminadas, resumen del dataset final**

---
## 4. Entrenamiento y evaluación de modelos

> ⚠️ **PENDIENTE — Juan Manuel copia aquí las celdas de validación cruzada, evaluación en test, matrices de confusión, curvas ROC y tabla comparativa de GridSearch del notebook 04 y 05**

---
## 5. Importancia de variables

Acá quise ver qué variables son las que más influyen en la predicción. Para eso usé dos métodos distintos y comparé los resultados.

El primero es la importancia que calcula Random Forest internamente mientras entrena. El segundo es lo que se llama permutation importance — básicamente desordenás los valores de una variable y medís cuánto empeoran las predicciones. Si empeoran mucho, esa variable era importante.

Usé los dos porque el primero puede favorecer variables que tienen muchas categorías distintas, entonces el segundo sirve para confirmar.

> Nota: que una variable sea importante no quiere decir que cause el riesgo. Solo está asociada estadísticamente en estos datos.

In [ ]:
# necesito esta función para que no haya problemas con columnas de tipo mixto
def convertir_a_texto(X):
    return X.astype(str)

# los mejores parámetros que encontró Juan Manuel con GridSearch
params = {
    'P1 Riesgo global': dict(n_estimators=200, max_depth=None, min_samples_leaf=1),
    'P2 Desaprobacion': dict(n_estimators=100, max_depth=None, min_samples_leaf=1),
    'P3 Abandono':      dict(n_estimators=200, max_depth=20,   min_samples_leaf=1),
}

problemas = [
    ('P1 Riesgo global', X1_tr, X1_te, y1_tr, y1_te),
    ('P2 Desaprobacion', X2_tr, X2_te, y2_tr, y2_te),
    ('P3 Abandono',      X3_tr, X3_te, y3_tr, y3_te),
]

print('Configuración lista.')

In [ ]:
# CELDA 3 — Función auxiliar: preprocesar + entrenar RF para un problema
#
# Clave: cada problema tiene su PROPIO preprocesador ajustado solo
# a su propio X_tr. Los datos transformados se guardan como arrays numpy
# separados del pipeline, eliminando el problema de incompatibilidad.

params = {
    'P1 Riesgo global': dict(n_estimators=200, max_depth=None, min_samples_leaf=1),
    'P2 Desaprobacion': dict(n_estimators=100, max_depth=None, min_samples_leaf=1),
    'P3 Abandono':      dict(n_estimators=200, max_depth=20,   min_samples_leaf=1),
}

problemas = [
    ('P1 Riesgo global', X1_tr, X1_te, y1_tr, y1_te),
    ('P2 Desaprobacion', X2_tr, X2_te, y2_tr, y2_te),
    ('P3 Abandono',      X3_tr, X3_te, y3_tr, y3_te),
]

# Guardar: modelo RF, datos transformados, nombres columnas originales
datos_por_problema = {}

for nombre, X_tr, X_te, y_tr, y_te in problemas:
    # Preprocesador propio para este problema
    prep = ColumnTransformer([
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('enc', OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False))]), cols_cat),
        ('num', SimpleImputer(strategy='most_frequent'), cols_num)
    ])

    # Transformar: fit SOLO en train, transform en train y test
    X_tr_t = prep.fit_transform(X_tr)
    X_te_t = prep.transform(X_te)        # usa los parámetros del train

    # Verificar consistencia ANTES de entrenar
    assert X_tr_t.shape[1] == X_te_t.shape[1], \
        f'ERROR: train {X_tr_t.shape[1]} cols vs test {X_te_t.shape[1]} cols'

    # Entrenar RF directamente sobre datos ya transformados
    rf = RandomForestClassifier(class_weight='balanced', random_state=42,
                                **params[nombre])
    rf.fit(X_tr_t, y_tr)

    auc = roc_auc_score(y_te, rf.predict_proba(X_te_t)[:,1])

    # Nombres de columnas después del one-hot (para mapear de vuelta)
    enc = prep.named_transformers_['cat']['enc']
    nombres_cols = enc.get_feature_names_out(cols_cat).tolist() + cols_num

    datos_por_problema[nombre] = {
        'rf': rf, 'prep': prep,
        'X_tr_t': X_tr_t, 'X_te_t': X_te_t,
        'X_te': X_te, 'y_te': y_te,
        'nombres_cols': nombres_cols, 'auc': auc
    }
    print(f'{nombre}: {X_tr_t.shape[1]} columnas codificadas | ROC-AUC test = {auc:.3f}')

print('\nTodos los modelos entrenados correctamente.')

In [ ]:
# CELDA 4 — Importancia interna de RF

importancias_internas = {}

for nombre, datos in datos_por_problema.items():
    rf           = datos['rf']
    nombres_cols = datos['nombres_cols']
    imp          = rf.feature_importances_

    # Agrupar por variable original sumando aportes de sus columnas one-hot
    imp_dict = {}
    for var_orig in X_base.columns.tolist():
        total = sum(v for n, v in zip(nombres_cols, imp)
                    if n.startswith(var_orig + '_') or n == var_orig)
        if total > 0:
            imp_dict[var_orig] = total

    imp_serie = pd.Series(imp_dict).sort_values(ascending=False)
    importancias_internas[nombre] = imp_serie

    print(f'{nombre} — Top 10:')
    print(imp_serie.head(10).round(4).to_string())
    print()

# Gráfico
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (nombre, imp_serie) in zip(axes, importancias_internas.items()):
    top15 = imp_serie.head(15)
    ax.barh(top15.index[::-1], top15.values[::-1], color='#3498DB', alpha=0.85)
    ax.set_title(f'{nombre}\n(importancia interna RF)', fontweight='bold')
    ax.set_xlabel('Importancia acumulada')
plt.suptitle('Top 15 — Importancia interna de Random Forest',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# CELDA 5 — Permutation importance MANUAL sobre datos ya transformados
#
# Ahora trabajamos directamente con arrays numpy (X_te_t)
# sin pasar por el pipeline → sin riesgo de incompatibilidad de columnas.
# Para cada variable original, desordenamos TODAS sus columnas one-hot juntas.

print('Calculando permutation importance (puede tardar 1-2 minutos)...\n')

N_REPEATS = 5
rng = np.random.RandomState(42)
perm_importancias = {}

for nombre, datos in datos_por_problema.items():
    rf           = datos['rf']
    X_te_t       = datos['X_te_t'].copy()
    y_te         = datos['y_te']
    nombres_cols = datos['nombres_cols']
    auc_base     = datos['auc']

    # Índices de columnas transformadas para cada variable original
    indices_var = {}
    for var_orig in X_base.columns.tolist():
        idx = [i for i, n in enumerate(nombres_cols)
               if n.startswith(var_orig + '_') or n == var_orig]
        if idx:
            indices_var[var_orig] = idx

    resultados = []
    for var_orig, idx in indices_var.items():
        caidas = []
        for _ in range(N_REPEATS):
            X_perm = X_te_t.copy()
            # Desordenar todas las columnas de esta variable juntas
            perm_idx = rng.permutation(X_perm.shape[0])
            X_perm[:, idx] = X_perm[perm_idx][:, idx]
            auc_perm = roc_auc_score(y_te, rf.predict_proba(X_perm)[:,1])
            caidas.append(auc_base - auc_perm)
        resultados.append({
            'variable':    var_orig,
            'importancia': np.mean(caidas),
            'std':         np.std(caidas)
        })

    perm_df = (pd.DataFrame(resultados)
                 .sort_values('importancia', ascending=False)
                 .reset_index(drop=True))
    perm_importancias[nombre] = perm_df

    print(f'{nombre} — Top 10 (caída en ROC-AUC):')
    print(perm_df[perm_df['importancia']>0].head(10)
          [['variable','importancia','std']].round(4).to_string(index=False))
    print()

In [ ]:
# CELDA 6 — Gráfico permutation importance
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (nombre, perm_df) in zip(axes, perm_importancias.items()):
    top15 = perm_df[perm_df['importancia']>0].head(15)
    ax.barh(top15['variable'][::-1], top15['importancia'][::-1],
            xerr=top15['std'][::-1], color='#E74C3C', alpha=0.85, capsize=4)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'{nombre}\n(permutation importance)', fontweight='bold')
    ax.set_xlabel('Caída en ROC-AUC al desordenar')
plt.suptitle('Top 15 — Permutation importance',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# CELDA 7 — Resumen consolidado para el informe
print('=' * 60)
print('VARIABLES MÁS RELEVANTES — RESUMEN PARA EL INFORME')
print('=' * 60)
print()
print('Variables en top 10 de AMBOS métodos por problema:')
print()

coincidencias_globales = {}
for nombre in datos_por_problema:
    top_int  = set(importancias_internas[nombre].head(10).index)
    top_perm = set(perm_importancias[nombre][perm_importancias[nombre]['importancia']>0]
                   .head(10)['variable'])
    coinciden = top_int & top_perm
    coincidencias_globales[nombre] = coinciden
    print(f'{nombre}:')
    for v in sorted(coinciden):
        print(f'  ✓ {v}')
    print()

# Variables que aparecen en los 3 problemas
todas = coincidencias_globales['P1 Riesgo global'] & \
        coincidencias_globales['P2 Desaprobacion'] & \
        coincidencias_globales['P3 Abandono']

print('Variables relevantes en los 3 problemas (más confiables):')
for v in sorted(todas):
    print(f'  ★ {v}')
print()
print('=' * 60)
print('ADVERTENCIA: asociación no implica causalidad.')

---
## 6. Ajuste del umbral de decisión

Algo que me pareció interesante es que el modelo no decide directamente si un estudiante está en riesgo o no — lo que hace es dar una probabilidad. Por ejemplo, puede decir que un estudiante tiene 0.38 de probabilidad de riesgo.

Por defecto, si esa probabilidad supera 0.50 se clasifica como riesgo. Pero ese corte en 0.50 no es el único posible. Si lo bajamos a 0.40, el modelo va a marcar como riesgo a más estudiantes — va a detectar más casos reales pero también va a tener más falsas alarmas.

En este caso me parece que vale la pena bajar el umbral, porque el costo de no detectar a un estudiante en riesgo es mayor que el costo de intervenir con alguien que en realidad iba a estar bien.

Probé los umbrales: 0.50, 0.45, 0.40, 0.35, 0.30 y 0.25.

In [ ]:
# CELDA 3 — Tabla de métricas por umbral para los 3 problemas

umbrales = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]
tablas   = {}

for nombre, _, _, _, _ in problemas:
    y_prob = modelos[nombre]['y_prob']
    y_te   = modelos[nombre]['y_te']

    filas = []
    for u in umbrales:
        y_pred = (y_prob >= u).astype(int)
        tp = int(((y_pred==1) & (y_te==1)).sum())
        fp = int(((y_pred==1) & (y_te==0)).sum())
        fn = int(((y_pred==0) & (y_te==1)).sum())
        filas.append({
            'Umbral':           u,
            'Recall':           round(recall_score(y_te, y_pred), 3),
            'Precision':        round(precision_score(y_te, y_pred, zero_division=0), 3),
            'F1':               round(f1_score(y_te, y_pred, zero_division=0), 3),
            'Detectados (TP)':  tp,
            'Falsas alarmas (FP)': fp,
            'No detectados (FN)':  fn,
        })

    tabla = pd.DataFrame(filas).set_index('Umbral')
    tablas[nombre] = tabla
    print(f'\n── {nombre} ──')
    print(tabla.to_string())

In [ ]:
# CELDA 4 — Gráfico: curvas de recall y precisión vs umbral

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (nombre, _, _, _, _) in zip(axes, problemas):
    tabla = tablas[nombre]
    ax.plot(tabla.index, tabla['Recall'],    'o-', color='#E74C3C',
            label='Recall (detección)', linewidth=2)
    ax.plot(tabla.index, tabla['Precision'], 's--', color='#3498DB',
            label='Precision', linewidth=2)
    ax.plot(tabla.index, tabla['F1'],        '^:', color='#2ECC71',
            label='F1-score', linewidth=2)
    ax.axvline(0.40, color='gray', linestyle='--', alpha=0.5, label='Umbral 0.40')
    ax.set_xlabel('Umbral de decisión')
    ax.set_ylabel('Métrica')
    ax.set_title(nombre, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.invert_xaxis()  # de 0.50 a 0.25 de izquierda a derecha
    ax.legend(fontsize=8)

plt.suptitle('Efecto del umbral sobre Recall, Precision y F1',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Eje X invertido: izquierda = umbral alto, derecha = umbral bajo')
print('Al bajar el umbral (derecha): recall sube, precision baja')

In [ ]:
# CELDA 5 — Resumen para el informe: umbral recomendado por problema

print('=' * 65)
print('ANÁLISIS DE UMBRAL — RECOMENDACIONES PARA EL INFORME')
print('=' * 65)

for nombre, _, _, _, _ in problemas:
    tabla  = tablas[nombre]
    y_prob = modelos[nombre]['y_prob']
    y_te   = modelos[nombre]['y_te']
    n_riesgo_real = int((y_te == 1).sum())

    print(f'\n{nombre}')
    print(f'Casos reales de riesgo en test: {n_riesgo_real}')
    print()

    # Umbral 0.50 (default)
    u50 = tabla.loc[0.50]
    print(f'  Umbral 0.50 (default):')
    print(f'    Detecta {u50["Detectados (TP)"]} de {n_riesgo_real} casos reales '
          f'(recall {u50["Recall"]:.0%})')
    print(f'    Genera {u50["Falsas alarmas (FP)"]} falsas alarmas')
    print(f'    Pierde {u50["No detectados (FN)"]} casos reales')

    # Umbral 0.40
    u40 = tabla.loc[0.40]
    print(f'\n  Umbral 0.40 (recomendado en el paper):')
    print(f'    Detecta {u40["Detectados (TP)"]} de {n_riesgo_real} casos reales '
          f'(recall {u40["Recall"]:.0%})')
    print(f'    Genera {u40["Falsas alarmas (FP)"]} falsas alarmas')
    print(f'    Pierde {u40["No detectados (FN)"]} casos reales')
    print(f'    Ganancia vs 0.50: '
          f'+{u40["Detectados (TP)"]-u50["Detectados (TP)"]} detecciones, '
          f'+{u40["Falsas alarmas (FP)"]-u50["Falsas alarmas (FP)"]} falsas alarmas')

print()
print('=' * 65)
print('CONCLUSIÓN:')
print('  El umbral 0.40 mejora el recall en todos los problemas')
print('  con un incremento moderado de falsas alarmas.')
print('  La decisión final corresponde a la institución según')
print('  el costo de intervenir vs. el costo de no detectar.')
print('=' * 65)

---
## 7. Conclusiones

---
## 7. Conclusiones

En base a los resultados obtenidos, me parece que el trabajo logró el objetivo principal que era ver si era posible predecir el riesgo académico con los datos que se tienen al inicio de la cursada.

Algunas cosas que me resultaron interesantes:

- Random Forest fue el que mejor resultados dio en los tres problemas. No esperaba que fuera tan consistente.
- El problema de abandono fue el más fácil de predecir (ROC-AUC 0.740). Creo que tiene sentido porque el abandono está más relacionado con factores socioeconómicos que sí están en el dataset.
- Cuando bajamos el umbral de 0.50 a 0.40, detectamos más estudiantes en riesgo sin aumentar demasiado las falsas alarmas. Me parece una buena opción para un sistema de alerta temprana.
- La variable `procedencia_localidad` aparece como importante en los tres problemas. Eso me hace pensar que la distancia a la facultad tiene algo que ver con el riesgo, aunque no podemos decir que sea la causa.

**Limitaciones que encontramos:**

El modelo solo usa datos del inicio de la cursada. No tiene información de cómo le fue en los parciales ni si asistió a clases. Creo que si tuviéramos eso los resultados mejorarían bastante.

También hay que tener cuidado con cómo se usan los resultados — el modelo no debería usarse para tomar decisiones automáticas sobre los estudiantes, sino como una herramienta para que los docentes puedan hacer un seguimiento.

**Una reflexión sobre las herramientas vistas en la materia:**

En la Unidad 4 vimos arquitecturas de deep learning como Transformers y mecanismos de atención, que son muy potentes para trabajar con texto o secuencias. En este trabajo decidimos usar modelos más clásicos como Random Forest y XGBoost porque el dataset es tabular y tiene menos de 1000 registros. Las redes profundas generalmente necesitan muchos más datos para funcionar bien. Creo que si tuviéramos los textos de los parciales o registros de actividad en el campus virtual, ahí sí podría tener sentido explorar algo con Transformers.

**Para seguir trabajando:**

- Transformar la variable de localidad en kilómetros a la sede (ya lo identificamos en el análisis pero no llegamos a implementarlo).
- Agregar datos de rendimiento durante la cursada cuando estén disponibles.
- Probar con más cohortes para ver si los resultados se mantienen.

---
## 8. Referencias

> ⚠️ **PENDIENTE — Juan Manuel y Stella: agregar referencias bibliográficas según las del Informefinal.pdf**